# Instruction Tuning: Making Models Follow Instructions

Pre-trained language models are next-token predictors — they complete text. Instruction tuning transforms them into assistants that follow user directives. This is the fine-tuning step that produces the gap between a base model (GPT-3) and a chat assistant (InstructGPT).

## The Pre-Instruction Tuning Problem

Base language models in 2020-2022 were powerful but awkward to use:
- Prompting required careful engineering; the model would complete your prompt in unexpected ways
- The model would generate text that continued the style of the prompt rather than answering it
- Safety and refusal behaviors were absent
- Multi-turn conversations were difficult to elicit consistently

Researchers found that fine-tuning on a dataset of (instruction, response) pairs — even a relatively small one — dramatically improved instruction following behavior. The model learned the format: when given an instruction, produce a helpful response.

## FLAN and the Multi-Task Instruction Tuning Insight

**FLAN** (Finetuned Language Net, Wei et al. 2021) was the first systematic study of instruction tuning. The key finding: fine-tuning on a diverse set of tasks with natural language instructions dramatically improved zero-shot performance on held-out tasks.

The diversity insight: a model instruction-tuned on 60+ tasks generalizes to new tasks far better than one tuned on a single task. The model learns the meta-skill of following instructions, not just the specific tasks.

**FLAN-T5** and **FLAN-PaLM** applied this at scale and established instruction tuning as a necessary component of capable models.

**Alpaca** (Stanford, 2023): 52,000 instruction-following examples generated by GPT-3.5 and used to fine-tune LLaMA. Showed that a small synthetic dataset could produce strong instruction following in open models.

In [ ]:
from dataclasses import dataclass
from typing import Callable
import random

@dataclass
class InstructionExample:
    instruction: str
    input_context: str
    output: str
    source: str = 'manual'

def format_alpaca(example: InstructionExample, add_eos: bool = True) -> str:
    if example.input_context:
        text = (
            f'### Instruction:\n{example.instruction}\n\n'
            f'### Input:\n{example.input_context}\n\n'
            f'### Response:\n{example.output}'
        )
    else:
        text = (
            f'### Instruction:\n{example.instruction}\n\n'
            f'### Response:\n{example.output}'
        )
    if add_eos:
        text += '</s>'
    return text

def format_chatml(example: InstructionExample) -> str:
    parts = ['<|im_start|>system\nYou are a helpful assistant.<|im_end|>']
    if example.input_context:
        user_msg = f'{example.instruction}\n\n{example.input_context}'
    else:
        user_msg = example.instruction
    parts.append(f'<|im_start|>user\n{user_msg}<|im_end|>')
    parts.append(f'<|im_start|>assistant\n{example.output}<|im_end|>')
    return '\n'.join(parts)

examples = [
    InstructionExample(
        instruction='Summarize the following text in one sentence.',
        input_context='The mitochondria is an organelle found in eukaryotic cells. It produces ATP through oxidative phosphorylation and is often called the powerhouse of the cell.',
        output='Mitochondria are eukaryotic organelles that produce ATP through oxidative phosphorylation.'
    ),
    InstructionExample(
        instruction='Write a Python function that returns the factorial of n.',
        input_context='',
        output='def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)'
    ),
]

for ex in examples:
    print('=== Alpaca format ===')
    print(format_alpaca(ex)[:200])
    print('\n=== ChatML format ===')
    print(format_chatml(ex)[:200])
    print()

## Data Quality vs. Quantity

A crucial finding from empirical instruction tuning research:

**LIMA** (Zhou et al. 2023): fine-tuned LLaMA-65B on only 1,000 carefully curated examples and produced results competitive with models fine-tuned on 50,000+ examples. The quality of instruction-response pairs mattered far more than the quantity.

Qualities of high-value instruction tuning data:
- **Diversity**: covers a wide range of task types, domains, and instruction styles
- **Difficulty**: includes hard, multi-step tasks that require reasoning
- **Output quality**: responses should demonstrate ideal behavior, not average behavior
- **Format variety**: different response structures, lengths, and styles

Low-quality data anti-patterns that reduce instruction following:
- Repetitive templates (e.g. all responses start with 'Certainly!')
- Sycophantic outputs that always agree with the user
- Short, low-information responses that satisfy the format but not the instruction

In [ ]:
def score_data_quality(examples: list) -> dict:
    if not examples:
        return {}
    # Diversity: unique first words of outputs
    first_words = [ex.output.split()[0].lower() if ex.output else '' for ex in examples]
    unique_starts = len(set(first_words))
    diversity_score = unique_starts / len(examples)
    # Sycophancy signal: common sycophantic openers
    sycophantic = ['certainly', 'great', 'of course', 'sure', 'absolutely', 'definitely']
    syco_count = sum(1 for ex in examples
                     if ex.output and ex.output.lower().split()[0] in sycophantic)
    syco_rate = syco_count / len(examples)
    # Average output length
    avg_len = sum(len(ex.output.split()) for ex in examples) / len(examples)
    # Task type diversity
    task_types = set()
    for ex in examples:
        inst = ex.instruction.lower()
        if any(w in inst for w in ['summarize', 'summary']):
            task_types.add('summarization')
        elif any(w in inst for w in ['write', 'generate', 'create']):
            task_types.add('generation')
        elif any(w in inst for w in ['explain', 'describe', 'what is']):
            task_types.add('explanation')
        elif any(w in inst for w in ['classify', 'categorize']):
            task_types.add('classification')
        else:
            task_types.add('other')
    return {
        'n_examples': len(examples),
        'diversity_score': round(diversity_score, 3),
        'sycophancy_rate': round(syco_rate, 3),
        'avg_output_words': round(avg_len, 1),
        'task_types': sorted(task_types),
    }

high_quality = [
    InstructionExample('Explain gradient descent', '', 'Gradient descent is an optimization algorithm that iteratively updates parameters in the direction of steepest loss decrease.'),
    InstructionExample('Summarize this article', 'The study found...', 'The research demonstrates that neural scaling laws hold across modalities.'),
    InstructionExample('Write a haiku about autumn', '', 'Crimson leaves descend / Silence fills the cooling air / Winter waits nearby'),
    InstructionExample('What causes inflation?', '', 'Inflation results from excess money supply relative to goods, supply chain disruptions, or demand shocks.'),
]
low_quality = [
    InstructionExample('Explain gradient descent', '', 'Certainly! Gradient descent is an algorithm.'),
    InstructionExample('Summarize this', 'text here', 'Certainly! Here is a summary.'),
    InstructionExample('Write something', '', 'Certainly! Here is something creative.'),
    InstructionExample('What is AI?', '', 'Certainly! AI is artificial intelligence.'),
]

print('High quality dataset:')
for k, v in score_data_quality(high_quality).items():
    print(f'  {k}: {v}')
print('\nLow quality dataset:')
for k, v in score_data_quality(low_quality).items():
    print(f'  {k}: {v}')

## The System Prompt and Multi-Turn Format

Modern instruction-tuned models are trained with a system prompt — a context that precedes the conversation and establishes the model's role and constraints. This enables:
- Persona customization without fine-tuning
- Safety and behavior constraints that persist across the conversation
- Task-specific configuration (output format, response length, expertise level)

The training data format matters: models trained on single-turn (instruction, response) pairs often struggle with multi-turn conversations. High-quality instruction tuning datasets include both single-turn and multi-turn examples with natural follow-up questions, clarification requests, and topic switches.